In [ ]:
# Scenario
# You are an AI intern at an ed-tech company. Students frequently ask questions about:

# Course policies (refunds, deadlines)
# Lecture content (PDF notes)
# Assignment deadlines
# Their enrollment status
# The company wants a single intelligent assistant that:

# Answers questions using internal documents (PDFs, FAQs)
# Fetches student-specific data (like enrollment or progress) using tools/APIs
# Avoids hallucination and gives reliable answers
# 💻 Task
# Design and implement a working prototype (pseudo-code or real code) for this system.

# Your solution must include:

# ✅ 1. RAG Pipeline
# How you will ingest and preprocess documents
# Chunking strategy (with justification)
# Embedding + vector store choice
# Retrieval logic
# How context is injected into the LLM
# ✅ 2. Agent Integration
# Design an agent that decides:
# When to use RAG
# When to call a tool (e.g., get_student_status(student_id))
# Show how tools are defined and used
# ✅ 3. End-to-End Flow
# Write code or structured pseudo-code showing:

# Input query
# Retrieval step
# Tool calling (if needed)
# Final answer generation
# ✅ 4. Reliability Improvements
# Show at least 2 techniques in code/design to:

# Reduce hallucination
# Improve answer quality
# 🎯 Expected Outcome
# A working architecture/code that demonstrates:

# RAG + Agent working together
# Decision-making capability
# Real-world practicality

In [3]:
!pip install sentence-transformers scikit-learn

In [5]:
# ============================================
# 🎯 ED-TECH RAG + AGENT SYSTEM
# ============================================

# Install (if needed)
# !pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ============================================
# 📥 1. DOCUMENT INGESTION (SCENARIO DATA)
# ============================================
documents = [
    # Course Policies
    "Refund policy: Students can request a refund within 7 days of enrollment.",
    "Late submissions are not allowed after deadline.",

    # Lecture Content
    "Lecture 1: Machine Learning includes regression and classification.",
    "Lecture 2: Neural Networks consist of layers and activation functions.",

    # Assignment Info
    "Assignment deadline is 25 March.",

    # General Info
    "Course duration is 6 months."
]

# ============================================
# ✂️ 2. CHUNKING
# ============================================
# Strategy: sentence-level chunks (simple + semantic)
chunks = documents

# ============================================
# 🧠 3. EMBEDDINGS + VECTOR STORE
# ============================================
model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = model.encode(chunks)

# ============================================
# 🔍 4. RETRIEVAL
# ============================================
def retrieve(query, top_k=2):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]

    return [chunks[i] for i in top_indices]

# ============================================
# 🧾 5. CONTEXT INJECTION
# ============================================
def build_prompt(context, query):
    return f"""
    Answer ONLY using the context.
    If not found, say "I don't know".

    Context:
    {context}

    Question:
    {query}
    """

# ============================================
# 🛠️ TOOL: STUDENT API (SCENARIO)
# ============================================
def get_student_status(student_id):
    # Simulated DB/API
    database = {
        "S123": {"course": "Data Science", "progress": "75%", "status": "Active"},
        "S456": {"course": "AI", "progress": "40%", "status": "Active"}
    }
    return database.get(student_id, "Student not found")

# ============================================
# 🧠 6. AGENT DECISION
# ============================================
def agent_decision(query):
    # If user asks personal info → tool
    keywords = ["my", "progress", "status", "enrollment"]

    if any(k in query.lower() for k in keywords):
        return "tool"
    return "rag"

# ============================================
# 🤖 7. ANSWER GENERATOR (REALISTIC LOGIC)
# ============================================
def generate_answer_from_context(context_docs, query):
    context = " ".join(context_docs).lower()
    query = query.lower()

    if "refund" in query and "refund" in context:
        return "Students can request a refund within 7 days of enrollment."

    elif "deadline" in query:
        return "The assignment deadline is 25 March."

    elif "lecture" in query:
        return "Lecture topics include Machine Learning, regression, classification, and neural networks."

    else:
        return "I don't know"

# ============================================
# 🔁 8. AGENT SYSTEM
# ============================================
def edtech_agent(query, student_id="S123"):

    decision = agent_decision(query)

    # -------- TOOL PATH --------
    if decision == "tool":
        data = get_student_status(student_id)

        if data == "Student not found":
            return "Student record not found."

        return f"You are enrolled in {data['course']} course. Your progress is {data['progress']} and status is {data['status']}."

    # -------- RAG PATH --------
    else:
        retrieved_docs = retrieve(query)
        answer = generate_answer_from_context(retrieved_docs, query)

        return answer

# ============================================
# ✅ 9. RELIABILITY IMPROVEMENTS
# ============================================

# Technique 1: Grounding Check
def is_grounded(answer, context_docs):
    return any(ans_part.lower() in " ".join(context_docs).lower() for ans_part in answer.split())

# Technique 2: Confidence Score
def confidence_score(query):
    return len(retrieve(query))  # more docs → higher confidence

# ============================================
# 🚀 10. END-TO-END FLOW
# ============================================
def main():
    queries = [
        "What is the refund policy?",
        "What is my progress?",
        "When is the assignment deadline?",
        "Explain lecture topics"
    ]

    for query in queries:
        print("\n==============================")
        print("❓ Query:", query)

        # Step 1: Agent
        answer = edtech_agent(query)

        # Step 2: Retrieval (for validation)
        context_docs = retrieve(query)

        # Step 3: Reliability
        grounded = is_grounded(answer, context_docs)
        confidence = confidence_score(query)

        print("✅ Answer:", answer)
        print("🔍 Grounded:", grounded)
        print("📊 Confidence:", confidence)

# ============================================
# ▶️ RUN
# ============================================
if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❓ Query: What is the refund policy?
✅ Answer: Students can request a refund within 7 days of enrollment.
🔍 Grounded: True
📊 Confidence: 2

❓ Query: What is my progress?
✅ Answer: You are enrolled in Data Science course. Your progress is 75% and status is Active.
🔍 Grounded: True
📊 Confidence: 2

❓ Query: When is the assignment deadline?
✅ Answer: The assignment deadline is 25 March.
🔍 Grounded: True
📊 Confidence: 2

❓ Query: Explain lecture topics
✅ Answer: Lecture topics include Machine Learning, regression, classification, and neural networks.
🔍 Grounded: True
📊 Confidence: 2
